In [ ]:
import glob
import sys
import os

import re


def extract_step(model_dir):
    """
    Extracts the training step number from a model directory name.
    Example:
        '/home/aiops/zhuty/litgpt_out/pretrain/tinyllama-mathpro-25ksteps-4nodes-2k/step-00020000_hf'
        -> 20000
    For directories not following the step pattern (e.g. 'final_hf'), returns None.
    """
    # Match step-000XXXXX pattern optionally ending with _hf or not
    match = re.search(r"(\d+)_hf", model_dir)
    if match:
        return int(match.group(1))
    else:
        return None


def extract_perf(result_dict):
    """
    Extract performance metrics from a result dictionary.
    
    Sample result dict:
    {
        'results': {
            'gsm8k': {
                'alias': 'gsm8k',
                'exact_match,strict-match': 0.10159,
                'exact_match_stderr,strict-match': 0.00832,
                'exact_match,flexible-extract': 0.10462,
                'exact_match_stderr,flexible-extract': 0.00843
            },
            'humaneval': {
                'alias': 'humaneval',
                'pass@1,create_test': 0.0,
                'pass@1_stderr,create_test': 0.0
            }
        }
    }
    """
    results = result_dict['results']
    final_dict = {}
    for task, task_result in results.items():
        alias = task_result.get('alias', task)
        for key, value in task_result.items():
            if key == 'alias':
                continue
            if 'stderr' in key:
                continue
            final_dict[f'{alias}_{key}'] = value
    return final_dict



In [ ]:
# Define multiple models for comparison
# Each entry: (base_path, factor, model_name)
models_config = [

       {
        "base_path": "/home/aiops/zhuty/nanotron/checkpoints",
        "name": "Llama3-1b + Math-Pro"
    },  
       {
        "base_path": "/home/aiops/zhuty/nanotron/checkpoints-mathpromaxsf-llama32-1b",
        "name": "Llama3-1b + Math-ProMax"
    },  
{
        "base_path": "/home/aiops/zhuty/nanotron/checkpoints-mathpros1sf-llama32-1b",
        "name": "Llama3-1b + Math-Pro-1%"
    },  

]
for config in models_config:
    if config["base_path"].endswith("2k") or '2k-const' in config["base_path"]:
        config["factor"] = 2
    elif config["base_path"].endswith("4k") or '4k-const' in config["base_path"]:
        config["factor"] = 4
    elif config["base_path"].endswith("8k") or '8k-const' in config["base_path"]:
        config["factor"] = 8
        if 'bsz512' in config["base_path"]:
            config["factor"] = 4
    else:
        config["factor"] = 4

In [ ]:
# Collect hf_dirs for all models
all_hf_dirs = {}
for config in models_config:
    base_path = config["base_path"]
    model_name = config["name"]
    hf_dirs = sorted(glob.glob(f"{base_path}/*hf"), key=lambda x: extract_step(x) if extract_step(x) is not None else -1)
    all_hf_dirs[model_name] = hf_dirs
    
print(all_hf_dirs)

In [ ]:
import json 

# Collect results for all models
# all_model_results is a dict: {model_name: [list of result dicts]}
all_model_results = {}

for config in models_config:
    model_name = config["name"]
    factor = config["factor"]
    hf_dirs = all_hf_dirs[model_name]
    
    model_dicts = []
    for dir in hf_dirs:
        final_dict = {}
        final_dict['model_name'] = model_name
        final_dict['factor'] = factor
        final_dict['checkpoint'] = dir.split('/')[-1]
        final_dict['step'] = extract_step(dir)
        if extract_step(dir) is None:
            continue

        for shot in [ '0shot','8shot','4shot']:
            result_file = glob.glob(dir + f"/harness_eval_{shot}/*/results*.json")
            if not len(result_file) > 0:
                continue
            assert len(result_file) > 0, f"The directory {dir} has more than one or no results.json file: {result_file}"
            result_file = result_file[0]
            with open(result_file, "r") as f:
                result = json.load(f)
            final_dict.update(extract_perf(result))

        model_dicts.append(final_dict)
    
    all_model_results[model_name] = model_dicts

print(f"Collected results for {len(all_model_results)} models: {all_model_results.keys()}")


In [ ]:
interested_metrics = [
    'gsm8k_exact_match,flexible-extract', 
    'hendrycks_math_exact_match,none',
    'humaneval_pass@1,create_test', 
    'arc_challenge_acc_norm,none', 
    'arc_easy_acc_norm,none', 
    'hellaswag_acc_norm,none'
]

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# Define line styles for different models
line_styles = ['-', '--', '-.', ':', (0, (3, 1, 1, 1)), (0, (5, 2))]
markers = ['o', 's', '^', 'D', 'v', 'p']

# Define colors for different tasks/metrics
colors = plt.cm.tab10.colors

plt.figure(figsize=(12, 6))

model_names = list(all_model_results.keys())

for model_idx, model_name in enumerate(model_names):
    model_dicts = all_model_results[model_name]
    
    # Get the factor for this model
    factor = model_dicts[0]['factor'] if model_dicts else 1
    
    # Sort by step
    model_dicts_sorted = sorted(model_dicts, key=lambda x: x['step'])
    
    steps = [d['step'] for d in model_dicts_sorted]
    tokens = [x * factor / 1000 for x in steps]
    
    linestyle = line_styles[model_idx % len(line_styles)]
    marker = markers[model_idx % len(markers)]
    
    for metric_idx, metric in enumerate(interested_metrics):
        values = [d.get(metric, None) for d in model_dicts_sorted]
        color = colors[metric_idx % len(colors)]
        
        # Filter out None values to keep lines connected
        valid_points = [(t, v) for t, v in zip(tokens, values) if v is not None]
        if valid_points:
            valid_tokens, valid_values = zip(*valid_points)
            plt.plot(valid_tokens, valid_values, marker=marker, linestyle=linestyle, 
                     color=color, markersize=5, linewidth=1.5)

# Create custom legend
# First: color legend for metrics
metric_handles = [Line2D([0], [0], color=colors[i % len(colors)], linewidth=2, 
                         label=metric.split(',')[0]) 
                  for i, metric in enumerate(interested_metrics)]

# Second: linestyle legend for models
model_handles = [Line2D([0], [0], color='black', linewidth=2, 
                        linestyle=line_styles[i % len(line_styles)],
                        marker=markers[i % len(markers)], markersize=5,
                        label=name) 
                 for i, name in enumerate(model_names)]

# Combine legends
first_legend = plt.legend(handles=metric_handles, title='Metrics', 
                          bbox_to_anchor=(1.05, 1), loc='upper left')
plt.gca().add_artist(first_legend)
plt.legend(handles=model_handles, title='Models', 
           bbox_to_anchor=(1.05, 0.5), loc='center left')

plt.xlabel("Tokens (B)")
plt.ylabel("Metric Value")
plt.title("Model Comparison: Metrics vs. Training Tokens")
plt.grid(True, alpha=0.3)
plt.tight_layout(rect=[0, 0, 0.75, 1])  # give space for legends
plt.show()
